In [2]:
import minari
dataset = minari.load_dataset('mujoco/halfcheetah/simple-v0', download=True)

In [3]:
import torch
for episode_data in dataset.iterate_episodes():
    observations = episode_data.observations
    observations = torch.tensor(observations)
    print(observations.shape)
    break

torch.Size([1001, 17])


In [12]:
env = dataset.recover_environment()
env.action_space.shape[0]

/home/haitong/anaconda3/envs/drones/lib/python3.10/site-packages/minari/dataset/minari_dataset.py:204: UserWarning: Installed mujoco version 3.3.0 does not meet the requirement ==3.2.3.
We recommend to install the required version with `pip install "mujoco==3.2.3"`
  warnings.warn(


6

In [5]:
env = dataset.recover_environment()

In [6]:
env.reset()

(array([-0.03019876, -0.0508219 ,  0.06213242,  0.09357072,  0.09359573,
        -0.08075633, -0.07568326, -0.09290087, -0.04226936,  0.08356013,
         0.0027152 , -0.05849673, -0.11837438,  0.22317824,  0.15275628,
        -0.14830531, -0.04747396]),
 {'x_position': -0.030782340129538996})

In [ ]:
import torch
import torch.nn.utils.rnn as rnn_utils
from torch.utils.data import DataLoader
import numpy as np

def collate_fn(batch, shuffle_trajectories=False):
    def map_fn(x):
        if shuffle_trajectories:
            return torch.as_tensor(np.random.permutation(x))
        else:
            return torch.as_tensor(x)
    return {
        "id": torch.Tensor([x.id for x in batch]),
        "observations": torch.nn.utils.rnn.pad_sequence(
            [map_fn(x.observations) for x in batch],
            batch_first=True
        ),
        "actions": torch.nn.utils.rnn.pad_sequence(
            [map_fn(x.actions) for x in batch],
            batch_first=True
        ),
        "rewards": torch.nn.utils.rnn.pad_sequence(
            [map_fn(x.rewards) for x in batch],
            batch_first=True
        ),
        "terminations": torch.nn.utils.rnn.pad_sequence(
            [map_fn(x.terminations) for x in batch],
            batch_first=True
        ),
        "truncations": torch.nn.utils.rnn.pad_sequence(
            [map_fn(x.truncations) for x in batch],
            batch_first=True
        )
    }

from functools import partial

dataloader = DataLoader(
    dataset, 
    batch_size=1, 
    shuffle=True, 
    collate_fn=partial(collate_fn, shuffle_trajectories=False),
    num_workers=4, 
    pin_memory=True # Set to True if you are training on a CUDA GPU
)

In [6]:
for batch in dataloader:
    print(batch['observations'][0][:5, 0])
    break

tensor([-0.0617, -0.0807, -0.1068, -0.1245, -0.1256], dtype=torch.float64)
